**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/yang-2025"], check=True)
    os.chdir("/content/encore/entries/yang-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings on dandiset 001212, checked again

The reconcile step of attempt `yang-2025-02` (docs/protocol.md, "Repeat attempts") checks here,
on DANDI:001212 version `0.241023.0931` (doi:10.48324/dandi.001212/0.241023.0931), the findings
of the earlier attempt `yang-2025-01` that `yang-2025-02`, made blind, did not report. Its other
findings are reported by `yang-2025-02` itself and are not repeated here. Only the files' metadata
and the epochs named in the checks are read, streamed from the archive with the helpers of
`yang.py`.

In [ ]:
import re
import collections
import pandas as pd
import yang


def text(v):
    """A string from an HDF5 scalar or one-element array of bytes."""
    return (v[0] if getattr(v, "shape", ()) else v).decode()


pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)
assets = yang.list_assets()
meta = []
for a in assets:
    with yang.open_asset(a["asset_id"]) as f:
        g = f["general"]
        el = g["extracellular_ephys/electrodes"]
        meta.append(dict(group=a["group"], rat=a["rat"], index=a["index"], kind=a["kind"], site=a["site"],
                         asset_id=a["asset_id"], start=text(f["session_start_time"][()])[:10],
                         epochs=list(f["acquisition"].keys()),
                         labels=tuple(x.decode() for x in el["group_name"][:]),
                         locations=tuple(x.decode() for x in el["location"][:]),
                         age=text(g["subject/age"][()]), weight=text(g["subject/weight"][()])))
meta = pd.DataFrame(meta)
print("dandiset", yang.DANDISET, "version", yang.VERSION, "|", len(meta), "files,",
      int(meta.epochs.apply(len).sum()), "epochs, rats", sorted(meta.rat.unique()))

## Earlier finding 3

> Sixteen clip names (date and seizure number) occur in two groups of the same rat with different
> contents, for example R1_2023_9_20-ictal1 to 3 in both R1 base3 and R1 DG-stim, and
> R3_2023_10_18-ictal1 and 10_20-ictal1 and 2 in both R3 DG-stim and CA1-stim. Some groups' clip
> dates fall outside their place in the order; R3's CA1-stim group (07) is dated 18 to 20 October,
> before its base3 group (06, from 22 October), and R5's base0 runs from 8 September to 25 October.

Epoch names are compared within each rat after removing the suffixes `-small` and `-large`; for
each name found in two groups, the two epochs' data are hashed (MD5 of the stored values) and their
lengths compared.

In [ ]:
def key(name):
    return re.sub(r"-(small|large)$", "", name)


rows = []
for _, m in meta.iterrows():
    for e in m.epochs:
        rows.append(dict(rat=m.rat, group=m.group, asset_id=m.asset_id, name=e, key=key(e), date=yang.epoch_date(e)))
ep = pd.DataFrame(rows)
two = ep.groupby(["rat", "key"]).group.nunique()
two = two[two > 1]
print("epoch names (suffixes removed) in more than one group of a rat:", len(two))
print("suffixes:", ep.name.str.extract(r"-(small|large)$")[0].value_counts().to_dict())
dup = ep.set_index(["rat", "key"]).loc[two.index].reset_index()
hashes = {}
for (aid, grp), g in dup.groupby(["asset_id", "group"]):
    with yang.open_asset(aid) as f:
        for n in g.name:
            d = f["acquisition"][n]["data"][:]
            hashes[(grp, n)] = (yang.md5(d), d.shape[0])
dup["md5"] = [hashes[(g, n)][0][:10] for g, n in zip(dup.group, dup.name)]
dup["samples"] = [hashes[(g, n)][1] for g, n in zip(dup.group, dup.name)]
print(dup[["rat", "key", "group", "name", "samples", "md5"]].to_string(index=False))
same = dup.groupby(["rat", "key"]).md5.nunique()
print("names whose two epochs hold the same data:", int((same == 1).sum()), "of", len(same))

In [ ]:
span = ep.groupby("group").date.agg(["min", "max", "count"])
span = meta.set_index("group")[["kind", "site"]].join(span)
print(span.loc[[g for g in span.index if g.startswith(("R3", "R5"))]].to_string())

## Earlier finding 6

> The electrodes table gives the ANT depth as DV -5.6 mm, where the paper gives -6.0 mm. Every
> session gives the subject's age as P2M and weight as 300-350 g, though a rat's sessions span up
> to four months.

Methods A gives DG AP -5.0, ML 3.2, DV -3.2; CA1 AP -4.2, ML 3.0, DV -2.2; CA3 AP -3.7, ML 4.0,
DV -2.8; ANT AP -1.2, ML 2.0, DV -6.0 (mm), and rats 7-9 weeks old and 250-300 g.

In [ ]:
PAPER = {"DG": (-5.0, 3.2, -3.2), "CA1": (-4.2, 3.0, -2.2), "CA3": (-3.7, 4.0, -2.8), "ANT": (-1.2, 2.0, -6.0)}
print("distinct electrode tables:", meta[["labels", "locations"]].drop_duplicates().shape[0])
for lab, loc in zip(meta.labels[0], meta.locations[0]):
    v = tuple(float(x) for x in re.findall(r"[-+]?\d+\.\d+", loc))
    site = lab.replace("R_", "")
    print(f"{lab}: {loc}; paper AP {PAPER[site][0]}, ML {PAPER[site][1]}, DV {PAPER[site][2]};",
          "matches" if v == PAPER[site] else "differs")
print("\nage and weight over the 56 files:", meta.groupby(["age", "weight"]).size().to_dict())
st = meta.groupby("rat").start.agg(["min", "max"])
st["days"] = (pd.to_datetime(st["max"]) - pd.to_datetime(st["min"])).dt.days
print("session start dates per rat:")
print(st.to_string())